# Dense Shelf Water densities and period means

We define Dense Shelf Water (DSW) each year as the water denser than the σ2 class with the largest surface water mass transformation (SWMT) on the shelf, and record that density along with the density of maximum cross-shelf export. We then average the spatial SWMT at the DSW density over 1900–2000 and 2090–2100. Inputs come from `01_prepare_dsw_wmt_budget` and `02_prepare_spatial_swmt`.

In [1]:
import sys

import numpy as np
import xarray as xr

sys.path.insert(0, "../src")
from paths import outputdir
from wmt import get_SWMT, select_from_location


reference_period = (1900, 2000)  # historical averaging window
future_period = (2090, 2100)  # end-of-century averaging window

abyssal_classes = slice(20, 38.1)  # sigma2 classes searched for the maxima
relative_tol = 2.5 / 100  # values smaller than this fraction of the year's extreme are treated as numerical noise
min_cell_swmt = 1100  # cells with |SWMT| below this [kg/s] are treated as numerical noise, not DSW outcrops

## Densities of maximum SWMT and export

In [2]:
ds = xr.open_zarr(outputdir("Southern_Ocean_DSW_1000m_WMT_Budget.zarr")).load()
ds = ds.groupby("time.year").mean("time")
ds["surface_boundary_fluxes"] = ds["boundary_fluxes"] - ds["bottom_flux_heat"]


def density_of_extreme(da, name, find_max=True):
    """Density of the SWMT maximum (DSW) or transport convergence minimum (export) each year, and the value there.

    Following Methods, the density is undefined (NaN) unless all denser classes have SWMT > 0 (DSW)
    or dPsi/dsigma2 > 0 (export, central differences). Only values larger in magnitude than
    `relative_tol` of that year's maximum are checked; smaller ones are treated as noise.
    """
    idx = da.argmax("sigma2_l_target") if find_max else da.argmin("sigma2_l_target")
    loc = da["sigma2_l_target"][idx]
    x = da if find_max else da.differentiate("sigma2_l_target")
    x = x.where(da["sigma2_l_target"] > loc)  # denser classes only
    checked = np.abs(x) > relative_tol * x.max("sigma2_l_target")
    loc = loc.where((x > 0).where(checked, True).all("sigma2_l_target")).rename(name)
    print(f"{name}: undefined in {int(loc.isnull().sum())} of {loc.size} experiment-years")
    return xr.merge([loc, select_from_location(da, loc)], compat="override")


swmt = ds["surface_boundary_fluxes"].sel(sigma2_l_target=abyssal_classes).fillna(0.0)
max_swmt = density_of_extreme(swmt, "max_surface_boundary_flux_density")
max_swmt["surface_boundary_fluxes"].attrs = {"units": "kg/s"}

export = ds["convergent_mass_transport"].sel(sigma2_l_target=abyssal_classes).fillna(0.0)
max_export = density_of_extreme(export, "max_export_density", find_max=False)
max_export["convergent_mass_transport"].attrs = {"units": "kg/s"}

max_swmt.to_zarr(outputdir("Southern_Ocean_DSW_1000m_max_SWMT.zarr"), mode="w")
max_export.to_zarr(outputdir("Southern_Ocean_DSW_1000m_max_export.zarr"), mode="w")

max_surface_boundary_flux_density: undefined in 0 of 500 experiment-years


/user/anthony.meza/CM4XAbyssalSWMT/notebooks/../src/wmt.py:127: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  results.append(xr.concat(exp_data, dim="year"))
/user/anthony.meza/CM4XAbyssalSWMT/notebooks/../src/wmt.py:127: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  results.append(xr.concat(exp_data, dim="year")

max_export_density: undefined in 0 of 500 experiment-years


/user/anthony.meza/CM4XAbyssalSWMT/notebooks/../src/wmt.py:127: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  results.append(xr.concat(exp_data, dim="year"))
/user/anthony.meza/CM4XAbyssalSWMT/notebooks/../src/wmt.py:127: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  results.append(xr.concat(exp_data, dim="year")

## Spatial SWMT at the DSW density, averaged by period

In [3]:
ds_sfc = xr.open_zarr(outputdir("Southern_Ocean_DSW_1000m_WMT_Surface_Fluxes_sigma_slice.zarr"))
swmt_sfc = get_SWMT(ds_sfc)
swmt_sfc["SWMT_residual"] = swmt_sfc["SWMT_heat_residual"] + swmt_sfc["SWMT_salt_residual"]

# drop the final year, as in our original analysis
dsw_density = max_swmt["max_surface_boundary_flux_density"].isel(year=slice(0, -1))
swmt_dsw = select_from_location(swmt_sfc.isel(year=slice(0, -1)), dsw_density, dim="sigma2_l", method="nearest")


def period_mean(period):
    mean = swmt_dsw.sel(exp="forced", year=slice(*period)).mean("year")
    return mean.where(np.abs(mean["SWMT"]) > min_cell_swmt).compute()


periods = [reference_period, future_period]
swmt_periods = xr.concat(
    [period_mean(p) for p in periods],
    dim=xr.IndexVariable("period", [f"{p[0]}-{p[1]}" for p in periods]),
)
for var in swmt_periods.data_vars:
    swmt_periods[var].attrs["units"] = "kg/s"

swmt_periods.to_zarr(outputdir("DSW_SWMT_periods.zarr"), mode="w")

/user/anthony.meza/CM4XAbyssalSWMT/notebooks/../src/wmt.py:127: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  results.append(xr.concat(exp_data, dim="year"))
/user/anthony.meza/CM4XAbyssalSWMT/notebooks/../src/wmt.py:127: FutureWarning: In a future version of xarray the default value for coords will change from coords='different' to coords='minimal'. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set coords explicitly.
  results.append(xr.concat(exp_data, dim="year")